# Week 1 · Day 2 — Lab 2
## Vectorization, ufuncs & Broadcasting

The number-one NumPy performance mistake is writing `for x in array`. NumPy is
fast because it pushes whole-array operations down into compiled C/SIMD kernels.
This lab is where you stop looping and start *thinking in arrays* — and where
**broadcasting** turns "different shapes" into clean, loop-free arithmetic.

You'll work with synthetic per-request telemetry from a fictional LLM gateway:
500 requests × 6 columns (latency, token counts, retrieved docs, cost,
top-1 similarity).

### Learning objectives
1. Replace a Python loop with a vectorized expression and **measure** the speedup.
2. Apply **ufuncs** (math functions and arithmetic operators) across whole arrays.
3. Apply the **broadcasting rules** — predict result shapes and diagnose shape errors.
4. **Z-score normalize** a feature matrix using per-column stats with `keepdims=True`.
5. Express conditional logic with `np.where` and `np.select` instead of loops.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** The performance gap | 12 min |
| **B.** Universal functions (ufuncs) | 10 min |
| **C.** Broadcasting rules & shape errors | 15 min |
| **D.** Z-score normalization | 15 min |
| **E.** `np.where` / `np.select` | 10 min |
| Wrap-up + stretch | 3 min |

### Files you need (in `data/`)
- `lab2_request_features.npy` — shape (500, 6), `float64`. Columns on deliberately
  different scales; column 0 (latency) contains a few negative noise readings.


In [ ]:
import time
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

# Solution is different here because of folder structure

DATA = Path("../data")
if not DATA.exists():
    DATA = Path(".")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# Per-request telemetry for a fictional LLM gateway. 500 rows, 6 columns.
features = np.load(DATA / "lab2_request_features.npy")
COLS = ["latency_ms", "input_tokens", "output_tokens",
        "retrieved_docs", "prompt_cost_usd", "similarity_top1"]
print("features:", features.shape, features.dtype)
for i, name in enumerate(COLS):
    print(f"  col {i}: {name}")

## Part A — The performance gap  *(guided)*

Let's feel the difference. We'll square-and-shift a million numbers two ways and
time both with `time.perf_counter`.


In [ ]:
big = features[:, 1].copy()              # input_tokens column
big = np.tile(big, 2000)                 # ~1,000,000 values to make timing visible
print("elements:", big.size)

# SLOW: explicit Python loop
t0 = time.perf_counter()
loop_out = np.empty_like(big)
for i in range(big.size):
    loop_out[i] = big[i] ** 2 + 2 * big[i] + 1
loop_secs = time.perf_counter() - t0

# FAST: vectorized expression — one compiled kernel, no Python per element
t0 = time.perf_counter()
vec_out = big ** 2 + 2 * big + 1
vec_secs = time.perf_counter() - t0

print(f"loop      : {loop_secs:.3f}s")
print(f"vectorized: {vec_secs:.4f}s")
print(f"speedup   : ~{loop_secs / max(vec_secs, 1e-9):.0f}x")
print("same result:", np.array_equal(loop_out, vec_out))

🧑‍🏫 **Instructor note — A.** Exact speedup varies by machine (commonly
50–300×); the *number* is not the point — the order of magnitude is. If timings
look noisy, mention `%timeit` in Jupyter (runs many loops, reports the best).
The key takeaway slide: *"if you're looping over elements, you're leaving
50–200× on the table."*


### Exercise A1 — Vectorize a cost-adjusted score
Compute, for every request, `score = similarity_top1 / (1 + prompt_cost_usd)`
(reward similarity, penalize cost). Do it **without a loop**, into
`adj_score` (shape `(500,)`). The cell also builds a slow loop reference so your
result can be checked for correctness.


In [ ]:
sim = features[:, 5]
cost = features[:, 4]

ref = np.empty(features.shape[0])
for i in range(features.shape[0]):
    ref[i] = sim[i] / (1 + cost[i])

adj_score = sim / (1 + cost)
print("first 5:", adj_score[:5].round(4))

In [ ]:
check("A1: adj_score has shape (500,)", lambda: adj_score.shape == (500,))
check("A1: adj_score matches the loop reference",
      lambda: np.allclose(adj_score, ref))

## Part B — Universal functions (ufuncs)

A **ufunc** is a compiled, element-wise function: `np.sqrt`, `np.exp`, `np.log`,
`np.sin`, ... and the arithmetic operators `+ - * /` are ufuncs too. They apply
to a whole array with no Python loop.


In [ ]:
v = np.array([1.0, 4.0, 9.0, 16.0])
print("sqrt:", np.sqrt(v))
print("log :", np.log(v).round(3))
print("a*b :", v * np.array([10, 10, 10, 10]))

### Exercise B1 — Stabilize token counts with `log1p`
Token counts span a wide range, so models often consume `log(1 + tokens)`.
`np.log1p` computes `log(1 + x)` accurately. Apply it to the `input_tokens`
column into `log_tokens`.


In [ ]:
log_tokens = np.log1p(features[:, 1])
print("first 5:", log_tokens[:5].round(3))
print("range:", log_tokens.min().round(3), "->", log_tokens.max().round(3))

In [ ]:
check("B1: log_tokens shape (500,)", lambda: log_tokens.shape == (500,))
check("B1: matches np.log1p of column 1",
      lambda: np.allclose(log_tokens, np.log1p(features[:, 1])))

## Part C — Broadcasting rules

Broadcasting lets NumPy combine arrays of different shapes without copying. The
rules, applied **right-to-left** across the shapes:
1. If one array has fewer dimensions, pad its shape on the **left** with 1s.
2. Two dimensions are compatible if they are **equal** or one of them is **1**.
3. A size-1 dimension is **stretched** to match the other.
4. If any dimension pair is incompatible (and neither is 1), NumPy **raises**.


In [ ]:
a = np.ones((3, 4))
row = np.array([1, 2, 3, 4])          # (4,) -> treated as (1, 4) -> stretched to (3, 4)
col = np.array([[10], [20], [30]])    # (3, 1) -> stretched to (3, 4)
print("a + row shape:", (a + row).shape)
print("a + col shape:", (a + col).shape)

### Exercise C1 — Apply per-column weights
You have a weight for each of the 6 columns. Multiply every row of `features`
by this `weights` vector using broadcasting (no loop), into `weighted`
(shape `(500, 6)`).


In [ ]:
weights = np.array([0.5, 1.0, 1.0, 2.0, 100.0, 10.0])
weighted = features * weights          # (500,6) * (6,) -> (500,6)
print("weighted shape:", weighted.shape)
print("row 0 before:", features[0].round(3))
print("row 0 after :", weighted[0].round(3))

In [ ]:
check("C1: weighted shape (500, 6)", lambda: weighted.shape == (500, 6))
check("C1: each column scaled by its weight",
      lambda: np.allclose(weighted[:, 4], features[:, 4] * 100.0))

### Exercise C2 — Make a shape error happen (on purpose)
Broadcasting *fails loudly* when shapes are incompatible. Try to add a length-5
vector to `features` (which has 6 columns). Catch the `ValueError` and set
`broadcast_failed = True` so you've proven the rule.


In [ ]:
broadcast_failed = False
try:
    _ = features + np.ones(5)     # (500,6) + (5,) -> incompatible
except ValueError as e:
    broadcast_failed = True
    print("Refused (as intended):", str(e).splitlines()[0])

In [ ]:
check("C2: incompatible broadcast raised ValueError",
      lambda: broadcast_failed is True)

🧑‍🏫 **Instructor note — C.** C1 reward: column 4 (cost) is now ×100 and column
0 (latency) is halved. C2: the error message names the shapes
("operands could not be broadcast together with shapes (500,6) (5,)"). Teach
them to *read the shapes in the error* — it's the fastest debugging signal in
NumPy. Note `(6,)` works but `(5,)` doesn't: rule 2 in action.


## Part D — Z-score normalization (the broadcasting payoff)

Normalizing each feature to mean 0 / std 1 is everyday ML preprocessing — and
it's pure broadcasting: subtract a per-column mean, divide by a per-column std.

Use `keepdims=True` so the stats keep shape `(1, 6)` instead of `(6,)`. The math
works either way, but `(1, 6)` makes the broadcast **explicit and
self-documenting** — the recommended production style.


In [ ]:
demo = np.array([[1.0, 10.0],
                 [2.0, 20.0],
                 [3.0, 30.0]])
mu = demo.mean(axis=0, keepdims=True)    # shape (1, 2)
sd = demo.std(axis=0, keepdims=True)     # shape (1, 2)
print("mean shape:", mu.shape, "-> per column:", mu.ravel())
print("normalized:\n", (demo - mu) / sd)

### Exercise D1 — Normalize the feature matrix
Compute `col_mean` and `col_std` over `features` with `axis=0, keepdims=True`
(both shape `(1, 6)`), then build `normalized = (features - col_mean) / col_std`.
After this, every column should have mean ≈ 0 and std ≈ 1.


In [ ]:
col_mean = features.mean(axis=0, keepdims=True)   # (1, 6)
col_std = features.std(axis=0, keepdims=True)     # (1, 6)
normalized = (features - col_mean) / col_std
print("col_mean shape:", col_mean.shape)
print("normalized column means:", normalized.mean(axis=0).round(6))
print("normalized column stds :", normalized.std(axis=0).round(6))

In [ ]:
check("D1: col_mean shape is (1, 6)", lambda: col_mean.shape == (1, 6))
check("D1: every column mean ~ 0", lambda: np.allclose(normalized.mean(axis=0), 0, atol=1e-9))
check("D1: every column std ~ 1", lambda: np.allclose(normalized.std(axis=0), 1, atol=1e-9))

🧑‍🏫 **Instructor note — D1.** Expected means ≈ 0 (to ~1e-16) and stds ≈ 1.
This is *exactly* how embedding inputs and eval-score distributions get
normalized later in the program — same three lines. If a std is 0 you'd get
NaNs; none of these columns are constant, so it's safe here. Flag for real data:
guard against zero-variance columns.


## Part E — Vectorized conditionals: `np.where` and `np.select`

`np.where(cond, a, b)` picks element-wise between `a` and `b`. It replaces a
loop with an `if/else` inside. `np.select` generalizes it to many conditions.


In [ ]:
x = np.array([-3.0, -1.0, 0.0, 2.0, 5.0])
print("clip negatives:", np.where(x < 0, 0.0, x))

### Exercise E1 — Clip negative latencies to zero
The latency column (index 0) has a few negative noise readings. Build
`latency_clipped` where negatives become `0.0` and everything else is unchanged,
using `np.where`.


In [ ]:
latency = features[:, 0]
latency_clipped = np.where(latency < 0, 0.0, latency)
n_fixed = int((latency < 0).sum())
print(f"clipped {n_fixed} negative readings")
print("min before:", latency.min().round(3), " min after:", latency_clipped.min().round(3))

In [ ]:
check("E1: no negative values remain", lambda: latency_clipped.min() >= 0.0)
check("E1: only the negatives changed",
      lambda: np.array_equal(latency_clipped[latency >= 0], latency[latency >= 0]))

### Exercise E2 — Tier requests by latency with `np.select`
Bucket each request into a latency tier:
- `"fast"`   when latency < 90
- `"normal"` when 90 ≤ latency < 160
- `"slow"`   otherwise

Use `np.select` with a list of conditions and a list of choices (default `"slow"`).
Apply it to `latency_clipped`.


In [ ]:
tiers = np.select(
    [latency_clipped < 90, latency_clipped < 160],
    ["fast", "normal"],
    default="slow",
)
uniq, counts = np.unique(tiers, return_counts=True)
print(dict(zip(uniq.tolist(), counts.tolist())))

In [ ]:
check("E2: tiers shape (500,)", lambda: tiers.shape == (500,))
check("E2: only the three expected labels appear",
      lambda: set(np.unique(tiers).tolist()) <= {"fast", "normal", "slow"})
check("E2: a fast request really is < 90",
      lambda: bool((latency_clipped[tiers == "fast"] < 90).all()))

🧑‍🏫 **Instructor note — E2.** `np.select` evaluates conditions in order and
takes the first True — that's why `< 90` then `< 160` cleanly partitions the
range. Contrast with `np.vectorize` (deck Part 7): it also produces labels but
calls Python per element and is *not* a speed tool. Reach for `np.where` /
`np.select` on hot paths.


## Stretch goals *(for fast finishers)*

**S1 — Outer product via broadcasting.** Build the 10×10 multiplication table
using only `np.arange` and broadcasting (a column vector times a row vector), no
loops.

**S2 — Pairwise differences.** Given the first 5 latencies, build a 5×5 matrix
`D` where `D[i, j] = latency[i] - latency[j]`, using broadcasting.


In [ ]:
col = np.arange(1, 11).reshape(10, 1)
row = np.arange(1, 11).reshape(1, 10)
mult_table = col * row
print("S1 mult_table[2, 3] =", mult_table[2, 3])   # 3 * 4 = 12

lat5 = features[:5, 0]
pairwise = lat5[:, None] - lat5[None, :]
print("S2 pairwise shape:", pairwise.shape)
print("S2 diagonal (should be 0s):", np.diag(pairwise))

In [ ]:
check("S1: 10x10 table, table[i,j] == (i+1)*(j+1)",
      lambda: mult_table.shape == (10, 10) and mult_table[9, 9] == 100)
check("S2: pairwise is 5x5 with zero diagonal",
      lambda: pairwise.shape == (5, 5) and np.allclose(np.diag(pairwise), 0))

## Wrap-up — what you can now do

- Replace element loops with vectorized expressions and explain the 50–200× gap.
- Use ufuncs (`sqrt`, `log1p`, operators) across whole arrays.
- Apply the broadcasting rules, predict result shapes, and read shape-mismatch errors.
- Z-score normalize a matrix with per-column stats and `keepdims=True`.
- Replace branching loops with `np.where` and `np.select`.

**Next:** Lab 3 — indexing and selection, where the *view vs. copy* distinction
from Lab 1 becomes a real production bug you'll learn to defuse.
